### **Data Aggregation**

**Goal**

Integrate multiple datasets from open-source, community-driven platforms to identify duplicate data center records, enrich existing information, and fill missing values using matched data from multiple sources.

In [97]:
import re
import pandas as pd
import numpy as np

from data_utils import (
    calculate_distance, 
    remove_city_state, 
    standardize_address
)

In [98]:
ft_data_centers = pd.read_csv('./data/raw/fractracker_data_centers.csv')
ft_crypto_centers = pd.read_csv('./data/raw/fractracker_crypto.csv')
atlas_data_centers = pd.read_csv('./data/raw/compute-atlas-facilities.csv')

print(f"FracTracker Data Centers {ft_data_centers.shape}")
print(f"FracTracker Crypto Centers {ft_crypto_centers.shape}")
print(f"Compute Atlas Data Centers {atlas_data_centers.shape}")

FracTracker Data Centers (1708, 53)
FracTracker Crypto Centers (302, 18)
Compute Atlas Data Centers (2394, 20)


**Standardize Columns**

In [99]:
ft_crypto_centers.columns = [
    re.sub(r'[?():]+', '', col.lower().replace(' ', '_'))
    for col in ft_crypto_centers.columns
]

atlas_data_centers = atlas_data_centers.rename(columns={'lon': 'long'})

In [100]:
atlas_cols = [
    'name', 'operator', 'facility_type', 'lat', 'long', 
    'capacity_operational_mw', 'capacity_planned_mw'
]

crypto_cols = [
    'name', 'address', 'city', 'state', 'lat', 'long',
    'company', 'probable_power_source', 'mw_nytimes'
]

atlas_data_centers = atlas_data_centers[atlas_cols]
ft_crypto_centers = ft_crypto_centers[crypto_cols]

#### **Aggregate Datasets**

Using a radius of approximately **0.75 miles** provides a good balance between identifying facilities across datasets and reducing the risk of incorrectly matching facilities that are simply located near one another.

**FracTracker Crypto Centers**

In [101]:
# Extract ZIP before removing it from the address
ft_crypto_centers["zip"] = (
    ft_crypto_centers["address"]
    .str.extract(r"(\d{5})\s*$")[0]
    .replace('', np.nan)
    .astype("Int64")
)

ft_crypto_centers["address"] = ft_crypto_centers.apply(
    lambda row: remove_city_state(
        row["address"],
        row["city"],
        row["state"]
    ),
    axis=1
)

# Standardize addresses
ft_crypto_centers["address"] = ft_crypto_centers["address"].apply(standardize_address)
ft_data_centers["address"] = ft_data_centers["address"].apply(standardize_address)

In [102]:
ft_crypto_centers["crypto_idx"] = ft_crypto_centers.index # add id field for merging
address_cols = ["address", "city", "state"]

# Exclude missing address
address_lookup = (
    ft_crypto_centers
    .dropna(subset=address_cols)[["address", "city", "state", "crypto_idx"]]
    .drop_duplicates(subset=address_cols)
)

# Find exact address matches
matched = pd.merge(
    ft_data_centers,
    address_lookup,
    on=address_cols,
    how="left"
)

address_matches = len(matched.loc[matched["crypto_idx"].notna()])

print(f"{len(matched.loc[matched["crypto_idx"].notna()])} matches found.")

19 matches found.


In [103]:
# Search for a crypto data centers within 0.075 miles
matched.loc[matched["crypto_idx"].isna(), "crypto_idx"] = calculate_distance(
    matched.loc[matched["crypto_idx"].isna()], 
    ft_crypto_centers,
    max_distance=0.075
)

merged_data_centers = pd.merge(
    matched,
    ft_crypto_centers.drop(columns="crypto_idx"),
    how="left",
    left_on="crypto_idx",
    right_index=True,
    suffixes=("", "_crypto")
)

print(f"{len(matched.loc[matched["crypto_idx"].notna()]) - address_matches} matches found.")

9 matches found.


**Compute Atlas Data Centers**

In [104]:
# Search for a atlas data centers within 0.075 miles
merged_data_centers['atlas_idx'] = calculate_distance(
    merged_data_centers, 
    atlas_data_centers, 
    max_distance=0.075
)

merged_data_centers = pd.merge(
    merged_data_centers, 
    atlas_data_centers, 
    how='left', 
    left_on='atlas_idx',
    right_index=True,
    suffixes=("", "_atlas")
)

print(f"{len(merged_data_centers.loc[merged_data_centers['atlas_idx'].notna()])} matches found.")

182 matches found.


#### **Missing Values**

In [105]:
def update_missing_rows(row):
    # Assign missing purpose
    if pd.isna(row["purpose"]) and pd.notna(row["facility_type"]):
        row["purpose"] = row["facility_type"]

    # Assign missing mw
    if pd.isna(row["mw"]):
        if pd.notna(row["mw_nytimes"]):
            row["mw"] = row["mw_nytimes"]
        elif pd.notna(row["capacity_operational_mw"]):
            row["mw"] = row["capacity_operational_mw"]
        elif pd.notna(row["capacity_planned_mw"]):
            row["mw"] = row["capacity_planned_mw"]

    # Assign missing power source
    if (
        pd.isna(row["power_source"])
        and pd.notna(row["probable_power_source"])
        and row["probable_power_source"] != "Unknown"
    ):
        row["power_source"] = row["probable_power_source"]

    return row


# Use datasets to fill in missing values
merged_data_centers = merged_data_centers.apply(update_missing_rows, axis=1)

In [107]:
merged_data_centers.info()

<class 'pandas.DataFrame'>
RangeIndex: 1708 entries, 0 to 1707
Data columns (total 72 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   OBJECTID                   1708 non-null   int64  
 1   GlobalID                   1708 non-null   str    
 2   facility_id                1708 non-null   str    
 3   facility_name              1708 non-null   str    
 4   address                    1663 non-null   str    
 5   city                       1704 non-null   str    
 6   state                      1708 non-null   str    
 7   zip                        1695 non-null   float64
 8   county                     1705 non-null   str    
 9   lat                        1708 non-null   float64
 10  long                       1708 non-null   float64
 11  status                     1708 non-null   str    
 12  location_confidence        1706 non-null   str    
 13  location_determination     1707 non-null   str    
 14  pur